In [ ]:
# notebook 07_clustering

import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt

adata = sc.read_h5ad("../data/interim/04_adata_responder_flagged.h5ad")
adata.X = adata.layers["lognorm"].copy()  # make sure X is the lognorm layer, explicitly
adata

In [ ]:
# ── highly variable gene (HVG) selection + save full gene set for marker lookups later ──
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor="seurat")
adata.raw = adata  # keep all genes accessible for rank_genes_groups later
adata = adata[:, adata.var.highly_variable].copy()
adata

In [ ]:
# ── Scale (Seurat-style: zero-center, unit variance, clip extreme outliers) ──
# z scores each gene: subtract mean, divide by std across cells, clips anything about 10 SD to 10
sc.pp.scale(adata, max_value=10)

In [ ]:
# ── PCA → neighbors → UMAP → Leiden ──
sc.tl.pca(adata, svd_solver="arpack", n_comps=50)
# build KNN graph in PCA space using first 30 of 50 computed PCs
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30)
# project PCs onto 2D space for plotting, stored in adata.obsm["X_umap"]
sc.tl.umap(adata)
# community-detection clustiering on KNN graph: obsp["connectivities"].  stored in adata.obs["leiden"] metadata
sc.tl.leiden(adata, resolution=1.0, flavor="igraph", n_iterations=2, key_added="leiden")

# how many cells per leiden (cluster)
adata.obs["leiden"].value_counts()

In [ ]:
# built in plotting fxn for umap
sc.pl.umap(adata, color="leiden", legend_loc="on data", legend_fontsize=6, frameon=False, title="Leiden clusters (res=1.0)")

In [ ]:
for cl in ["21", "22", "15"]:
    print(cl, adata.obs.loc[adata.obs["leiden"] == cl, "target"].value_counts().head(3))

In [ ]:
# marker testing
# adata.raw still has lognorm values for all genes (snapshotted before HVG subsetting + scaling)
sc.tl.rank_genes_groups(adata, groupby="leiden", method="wilcoxon", use_raw=True)

In [ ]:
# quick per-cluster panel of top marker genes
sc.pl.rank_genes_groups(adata, n_genes=15, sharey=False)

In [ ]:
lineage_markers = {
    "erythroid": ["HBB", "HBA1", "HBA2", "GYPA", "KLF1", "GATA1", "ALAS2"],
    "granulocyte/myeloid": ["ELANE", "MPO", "CEBPE", "ITGAM", "CSF3R"],
    "megakaryocyte": ["PF4", "ITGA2B", "GP9", "VWF", "PLEK"],
}

for cluster in adata.obs["leiden"].cat.categories:
    top_genes = sc.get.rank_genes_groups_df(adata, group=cluster).head(10)["names"].tolist()
    print(cluster, top_genes)

In [ ]:
# now we have marker genes for each cluster.
# use some GO and other similar resources to help annotate clusters by marker genes (computationally, first pass)

import gseapy as gp

# See what gene-set libraries are available (there are hundreds)
# gp.get_library_name(organism='Human')[:20]

# this is pinging 24 requests to API - need a time delay
import time

results = {}
for cluster in adata.obs["leiden"].cat.categories:
    top_genes = sc.get.rank_genes_groups_df(adata, group=cluster).head(25)["names"].tolist()
    
    for attempt in range(3):
        try:
            enr = gp.enrichr(
                gene_list=top_genes,
                gene_sets=['PanglaoDB_Augmented_2021', 'GO_Biological_Process_2021'],
                organism='human',
                outdir=None,
            )
            results[cluster] = enr.results.sort_values("Adjusted P-value").head(3)
            break
        except Exception as e:
            print(f"cluster {cluster} attempt {attempt+1} failed: {e}")
            time.sleep(5)  # back off before retrying
    
    time.sleep(2)  # be polite between clusters regardless of success

for cluster, top_hits in results.items():
    print(f"\n=== Cluster {cluster} ===")
    print(top_hits[["Gene_set", "Term", "Adjusted P-value", "Genes"]].to_string(index=False))

In [ ]:
# use PanglaoDB hits (cell-type specific) over GO terms.
# this is maybe faulty, as the marker genes were determined internally, for K562 line - limited in "cell type it can land on"

suggested_labels = {}
for cluster, top_hits in results.items():
    panglao_hits = top_hits[top_hits["Gene_set"] == "PanglaoDB_Augmented_2021"]
    if len(panglao_hits) > 0 and panglao_hits.iloc[0]["Adjusted P-value"] < 0.05:
        suggested_labels[cluster] = panglao_hits.iloc[0]["Term"]
    else:
        suggested_labels[cluster] = top_hits.iloc[0]["Term"]  # fall back to top GO term

for cluster, label in suggested_labels.items():
    print(cluster, "->", label)

In [ ]:
# many of these labels don't make sense for cells DE from K562 cells (myeloid)
# human review and reannotation below:

cluster_labels = {
    "0": "Myeloid-like 1",
    "1": "Myeloid-like 2",
    "2": "Ribosomal/Translation 1",
    "3": "Proteasome/Housekeeping",
    "4": "Cell cycle 1",
    "5": "Ambiguous 1",
    "6": "Erythroid 1",
    "7": "Erythroid 2",
    "8": "Erythroid 3 (ribosomal-high)",
    "9": "Cell cycle 2",
    "10": "Ribosomal/Translation 2",
    "11": "Cell cycle 3",
    "12": "Erythroid 4",
    "13": "Myeloid-like 3",
    "14": "Stress/immediate-early",
    "15": "Erythroid 5 (progenitor-like)",
    "16": "Erythroid 6",
    "17": "Erythroid 7 (heme/redox)",
    "18": "ER stress response",
    "19": "Ambiguous 2",
    "20": "Mitochondrial/OXPHOS",
    "21": "Erythroid 8 (progenitor-like)",
    "22": "Interferon response 3",
    "23": "Interferon response 2",
}

adata.obs["clusterName1"] = adata.obs["leiden"].map(cluster_labels).astype("category")
adata.obs["clusterName1"].value_counts()

In [ ]:
# plot UMAP with clusterName1 labeling, and save

from adjustText import adjust_text

import numpy as np
import matplotlib
# matplotlib.rcParams["font.family"] = "Arial"
import matplotlib.pyplot as plt
from matplotlib import patheffects as pe

import matplotlib.colors as mcolors
import colorsys

# fxn to darken colors used for cluster scatter, for labeling
def darken(hex_color, factor=0.8):
    """factor < 1 darkens; e.g. 0.6 = 60% of original brightness"""
    r, g, b = mcolors.to_rgb(hex_color)
    h, s, v = colorsys.rgb_to_hsv(r, g, b)
    v *= factor
    return colorsys.hsv_to_rgb(h, s, v)

# plot code
fig, ax = plt.subplots(figsize=(10, 9))

ax.scatter(
    umap_coords[:, 0], umap_coords[:, 1],
    c=point_colors, s=6, alpha=0.5, linewidths=0,
)

texts = []
for leiden_cluster in adata.obs["leiden"].cat.categories:
    mask = (adata.obs["leiden"] == leiden_cluster).values
    if mask.sum() == 0:
        continue
    cx, cy = umap_coords[mask, 0].mean(), umap_coords[mask, 1].mean()
    name = adata.obs.loc[mask, "clusterName1"].iloc[0]
    color = color_map[name]
    dark_color = darken(color, factor=0.55)

    ax.scatter(cx, cy, color=color, s=180, edgecolor="black", linewidth=0.8, zorder=5)

    t = ax.text(
        cx, cy, name,
        fontsize=11, fontweight="bold", ha="center", va="center", zorder=6,
        color=dark_color,
        path_effects=[pe.withStroke(linewidth=2.5, foreground="white")],
    )
    texts.append(t)

adjust_text(
    texts,
    ax=ax,
    expand_points=(1.5, 1.5),
    arrowprops=dict(arrowstyle="-", color="gray", lw=0.6, alpha=0.7),
)

ax.set_xlabel("UMAP1")
ax.set_ylabel("UMAP2")
ax.set_xticks([])
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(False)

ax.set_title("UMAP on Leiden clusters:\nlabeled via marker genes + Enrichr enrichment)", fontsize=12)

fig.tight_layout()
fig.savefig("../results/figures/UMAP_clusterNames1.png", dpi=200, bbox_inches="tight")
plt.show()